# Your own restaurant agent

You have seen the restaurant's agent in MLflow. Now the agent runs **in this
notebook**: `question → LangGraph model node → optional MCP tool → model → answer`.
It calls the provided model directly and logs to **restaurant-agent-playground**
in your existing workspace. The restaurant UI continues using its own service
and experiment. Your notebook edits do not change its system prompt or NeMo policy.

The notebook uses read-only menu/wait tools. The restaurant's NeMo shield is **not**
in this separate agent's model path. No package installation or pasted key is needed.
Run the setup and function cells once, then edit the controls and rerun the final cell.


In [ ]:
import os
import uuid
from typing import Annotated, TypedDict

import mlflow
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from helpers.mcp_client import call_read_tool, list_read_tools

for name in ("MLFLOW_TRACKING_URI", "MLFLOW_WORKSPACE", "MODEL_BASE_URL", "MODEL_NAME"):
    assert os.environ.get(name), f"Missing {name}; ask your instructor to check the workbench"
assert os.environ.get("OPENAI_API_KEY"), "Model credentials should be supplied by the workbench"

mlflow.tracing.disable_notebook_display()
mlflow.set_experiment("restaurant-agent-playground")
print("Workspace:", os.environ["MLFLOW_WORKSPACE"])
print("Experiment: restaurant-agent-playground")
print("Model:", os.environ["MODEL_NAME"])
# Never print the environment or API key. MLflow auth is injected by OpenShift AI.


## 1. Controls: change one thing at a time

Start with autologging and manual spans both enabled. Set `USE_MCP=False` to
remove tool discovery and execution entirely. The model then has no live menu
access; compare its answer rather than assuming it will admit this limitation.
Every run starts with a fresh conversation. Keep the question fixed for comparisons.


In [ ]:
LABEL = "first-run"  # Appears as playground.label on the trace
QUESTION = "Which vegetarian main dishes are at Sakura? Please check the menu."
USE_MCP = True
AUTOLOG = True
MANUAL_SPANS = True
TEMPERATURE = 0.2
MAX_MODEL_CALLS = 3  # Stop a tool loop instead of spending unbounded model quota
SYSTEM_PROMPT = """You help guests at Sakura restaurant (slug sakura).
Use the available tools for live menu and wait information. Never invent facts
that you cannot check. If no suitable tool is available, explain that limitation.
Keep your answer short and distinguish tool evidence from assumptions."""


## 2. Build a small LangGraph agent

The model decides whether to request a tool. The graph runs requested tools,
adds their results to the conversation, and calls the model again. Inspect and
edit these functions: this is the actual agent, not an HTTP driver for the app.

`mlflow.langchain.autolog()` records framework/model execution. A manual `TOOL`
span adds the MCP request's arguments/result and its boundary duration; ordinary
HTTP calls are not automatically traced by the LangChain integration.


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    model_calls: int
    tool_calls: int


def build_agent():
    extra_body = ({"chat_template_kwargs": {"enable_thinking": False}}
                  if os.environ.get("MODEL_EXTRA_BODY", "false").lower() == "true" else {})
    model = ChatOpenAI(
        model=os.environ["MODEL_NAME"], base_url=os.environ["MODEL_BASE_URL"],
        api_key=os.environ["OPENAI_API_KEY"], temperature=TEMPERATURE,
        max_tokens=700, timeout=60, max_retries=0, extra_body=extra_body,
    )
    tool_specs = list_read_tools(os.environ["MCP_URL"]) if USE_MCP else []
    allowed_tools = {tool["function"]["name"] for tool in tool_specs}
    chat_model = model.bind_tools(tool_specs) if tool_specs else model

    def call_model(state):
        if mlflow.get_current_active_span() is not None:
            mlflow.update_current_trace(tags=trace_tags())
        reply = chat_model.invoke([SystemMessage(content=SYSTEM_PROMPT), *state["messages"]])
        return {"messages": [reply], "model_calls": state["model_calls"] + 1}

    def call_tools(state):
        calls = state["messages"][-1].tool_calls
        if state["tool_calls"] + len(calls) > 4:
            raise RuntimeError("Playground tool-call limit reached; simplify the question")
        results = []
        for call in calls:
            if call["name"] not in allowed_tools:
                raise ValueError(f"Tool was not offered: {call['name']}")
            if MANUAL_SPANS:
                with mlflow.start_span(name=f"mcp.{call['name']}", span_type="TOOL") as span:
                    span.set_inputs(call["args"])
                    output = call_read_tool(os.environ["MCP_URL"], call["name"], call["args"])
                    span.set_outputs(output)
            else:
                output = call_read_tool(os.environ["MCP_URL"], call["name"], call["args"])
            results.append(ToolMessage(content=output, tool_call_id=call["id"], name=call["name"]))
        return {"messages": results, "tool_calls": state["tool_calls"] + len(calls)}

    def next_step(state):
        if not state["messages"][-1].tool_calls:
            return END
        if state["model_calls"] >= MAX_MODEL_CALLS:
            raise RuntimeError("Model-call limit reached; simplify the question or raise the limit")
        return "tools"

    graph = StateGraph(AgentState)
    graph.add_node("model", call_model)
    graph.add_node("tools", call_tools)
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", next_step, {"tools": "tools", END: END})
    graph.add_edge("tools", "model")
    return graph.compile()


## 3. Add your own tracing boundary

The manual root names one complete question/answer. The decorator on
`format_answer` shows how to add a span around your own Python function.
Try changing that function to add a visible output-format check, then inspect
its inputs/outputs. A custom span adds application meaning; it complements the
framework spans rather than replacing them.

`run_agent` rebuilds the graph from your current controls each time. Toggling
`AUTOLOG` explicitly enables **or disables** autologging in this kernel.


In [ ]:
def trace_tags():
    return {"playground.label": LABEL, "playground.mcp": str(USE_MCP),
            "playground.autolog": str(AUTOLOG), "playground.manual": str(MANUAL_SPANS),
            "playground.temperature": str(TEMPERATURE)}


@mlflow.trace(name="format_answer", span_type="CHAIN")
def traced_format_answer(text):
    return format_answer(text)


def format_answer(text):
    # Quick win: add your own formatting/validation logic here.
    return str(text).strip()


def run_agent(question):
    mlflow.langchain.autolog(disable=not AUTOLOG, run_tracer_inline=True)
    graph = build_agent()

    def turn(question):
        if mlflow.get_current_active_span() is not None:
            mlflow.update_current_trace(tags={**trace_tags(), "playground.turn": str(uuid.uuid4())})
        state = graph.invoke({"messages": [HumanMessage(content=question)],
                              "model_calls": 0, "tool_calls": 0},
                             config={"recursion_limit": 2 * MAX_MODEL_CALLS + 2})
        raw = state["messages"][-1].content
        answer = traced_format_answer(raw) if MANUAL_SPANS else format_answer(raw)
        return {"answer": answer, "model_calls": state["model_calls"],
                "tools": [message.name for message in state["messages"] if isinstance(message, ToolMessage)]}

    before = mlflow.get_last_active_trace_id()
    invoke = mlflow.trace(turn, name="playground.turn", span_type="AGENT") if MANUAL_SPANS else turn
    result = invoke(question)
    mlflow.flush_trace_async_logging()
    after = mlflow.get_last_active_trace_id()
    result["trace_id"] = after if after != before else None
    return result


## 4. Run one question, then look in MLflow

Set `RUN_AGENT=True` and run the next cell. It makes real model calls and, when
enabled, MCP requests. Reset it to `False` afterwards to avoid accidental calls
on **Run all**. Open **restaurant-agent-playground → Traces** in your MLflow
workspace and find the printed trace ID or `playground.label` tag.


In [ ]:
RUN_AGENT = False  # Set True to send QUESTION through your own agent
if RUN_AGENT:
    result = run_agent(QUESTION)
    print(result["answer"])
    print("Model calls:", result["model_calls"], "| Tools:", result["tools"])
    print("Trace:", result["trace_id"] or "No new trace (check your tracing controls)")
else:
    print("No model calls made. Set RUN_AGENT=True when ready.")


## 5. Pick an experiment

Change the controls cell, then rerun **only the final execution cell**. Change
`LABEL` each time and compare traces. You do not need to redefine the functions
unless you edit their code.

| Change | What to investigate |
|---|---|
| `AUTOLOG=True`, `MANUAL_SPANS=True` | Root + graph/model spans + explicit MCP/format spans |
| `AUTOLOG=True`, `MANUAL_SPANS=False` | Graph/model spans remain; your custom root and MCP/format boundaries disappear |
| `AUTOLOG=False`, `MANUAL_SPANS=True` | Custom root/MCP/format spans remain; framework/model internals disappear |
| Both tracing controls `False` | The agent still answers; this kernel creates no trace for the call |
| `USE_MCP=False` | No MCP discovery/execution; compare grounding, tool spans and number of model calls |
| Ask about wait time instead of menu | A different tool/arguments/result, if the model chooses it |
| Edit `SYSTEM_PROMPT` to request three short sections | Prompt affects this notebook agent only; inspect its model input |
| Add logic to `format_answer`, or move/add a manual span | Decide which application step needs its own observable boundary |

For a quick core exercise, compare **both → autolog only → manual only**, then
disable MCP for the same menu question. Extra runs are open exploration; share
provider quota with the room. Only one cell runs a question; no automatic batch.
There is no conversation memory between runs. A model/tool timeout or loop limit
raises an error; inspect the failed trace rather than treating it as success.

**Finish:** set `RUN_AGENT=False`. Keep your traces. Restore the restaurant's
prompt/empty policy if needed; notebook controls do not modify either.

References: [MLflow LangGraph tracing](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langgraph/),
[LangGraph graph quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart).
